In [3]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [ ]:
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"]
os.environ["WATSONX_APIKEY"] 
os.environ["PINECONE_API_KEY"]


In [5]:
from langchain_ibm import ChatWatsonx

model = ChatWatsonx(
    model_id="meta-llama/llama-3-1-8b-instruct",
    url="https://eu-de.ml.cloud.ibm.com",
    project_id="aeabba3a-e8d1-4181-bcff-d0a1b639e896"
)

/Users/sathvika/MCT/SEM-4/industry-project/TAL_Chatbot/.conda/lib/python3.11/site-packages/ibm_watsonx_ai/foundation_models/utils/utils.py:436: LifecycleWarning: Model 'meta-llama/llama-3-1-8b-instruct' is in deprecated state from 2025-01-22 until 2025-05-30. IDs of alternative models: llama-3-2-11b-vision-instruct. Further details: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-model-lifecycle.html?context=wx&audience=wdp
  warn(model_state_warning, category=LifecycleWarning)


In [11]:
# from langchain_ibm import WatsonxEmbeddings

# embeddings = WatsonxEmbeddings(
#     model_id="ibm/slate-125m-english-rtrvr",
#     url="https://eu-de.ml.cloud.ibm.com",
#     project_id="aeabba3a-e8d1-4181-bcff-d0a1b639e896",
# )

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="hkunlp/instructor-large")

In [13]:
vectors = embeddings.embed_query("How are you")
len(vectors)


768

In [14]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"], 
              environment="us-east-1")
index_name = "tal-chatbot"
index = pc.Index(index_name)

vector_store = PineconeVectorStore(embedding=embeddings, index=index)

Loading Documents


In [15]:
from langchain_community.document_loaders import JSONLoader
file_path = "/Users/sathvika/MCT/SEM-4/industry-project/TAL_Chatbot/DataPrep/converters.json"

loader = JSONLoader(
    file_path=file_path,
    jq_schema='.[]',  # This will iterate over each item
    text_content=False  # We'll use the entire object as content
)

documents = loader.load()


In [18]:
len(documents)

65

Uploading Embeddings


In [20]:
vectorstore = PineconeVectorStore.from_documents(
    documents,
    embedding=embeddings,
    index_name=index_name
)

Consine Sim Retrieval

In [21]:
from langchain import hub
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

# Define prompt for question-answering
prompt = hub.pull("rlm/rag-prompt")


# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = model.invoke(messages)
    return {"answer": response.content}


# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [26]:
response = graph.invoke({"question": "What all convertors can i use for *MIX 6 halosphere?"})
print(response["answer"])

You can use the following converters for *MIX 6 halosphere: 
- Powerled Converter Remote 260mA 20W IP20 Mains Dim LC (ARTNR: 930604) 
- Powerled Converter Remote 260mA 12W IP20 1-10V (ARTNR: 930599) 
- Powerled Converter Remote 180mA 8W IP20 1-10V (ARTNR: 930546)
